<a href="https://colab.research.google.com/github/divyaddot7-wq/AI-study-Buddy/blob/main/DIVYA_RAG_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# STEP 0: Install
!pip install -q pymupdf sentence-transformers scikit-learn google-generativeai

# STEP 0.1: Create PDF automatically
import fitz
doc = fitz.open()
all_pages =  [
"""SRM Hostel Rulebook 2025-26 - Kaveri Block-C. Valid June 1 2025 to May 31 2026. Admission fee Rs. 5,000 non-refundable. Annual fee for 3-sharing AC room is Rs. 1,25,000 including food and maintenance. Annual fee for 3-sharing non-AC is Rs. 95,000. 2-sharing AC is Rs. 1,55,000. Mess: Breakfast 7-9 AM, Lunch 12:30-2:30 PM, Snacks 5-6 PM, Dinner 7:30-9 PM. Wastage fine Rs. 100.""",

"""Hostel Timings: Gate 5:00 AM to 9:30 PM weekdays, 10:00 PM weekends. Biometric 9:15-9:30 PM daily. Miss 3 days biometric = warning. Late entry after 11 PM fine Rs. 500 per instance. After 12 AM fine Rs. 1000 and parent call. Room allotment date is July 15th 2025. Reporting date July 20-25. Visitors: Male visitors not allowed inside rooms only lounge. Timing 4:30-6:30 PM. ID needed.""",

"""Rules: Ragging strictly prohibited. Fine Rs. 10,000 and expulsion. Smoking, alcohol, drugs prohibited. Room inspection weekly. Electrical appliances not allowed except laptop, mobile charger. Fee for 2-sharing room in Harvard? This document is for SRM Hostel only, not Harvard. Principal of SRM University info not in hostel rulebook. Hostel fee for boys hostel is same as above. Girls hostel separate block. Study hours 10 PM to 12 AM silent hours.""",

"""Outpass: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warden approval. Medical facility 24/7. Ambulance available. Hostel fee includes medical insurance upto Rs. 50,000. AC timing 9 PM to 6 AM. Extra AC hours charged Rs. 50 per hour. Laundry Rs. 1000 per semester optional. WiFi 100 Mbps per room."""
]

for t in all_pages:
    p = doc.new_page(width=595, height=842); p.insert_text((50,50), t, fontsize=11, fontname="helv")
doc.save("/content/SRM_Hostel_Rulebook_2025.pdf")
print("PDF created")

# STEP 1 & 2: Read and Chunk
import fitz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

pdf = fitz.open("/content/SRM_Hostel_Rulebook_2025.pdf")
full_text = "".join([page.get_text() for page in pdf])
print(f"Total characters: {len(full_text)}")

def chunk_text(text, size=800, overlap=150):
    chunks=[]; s=0
    while s < len(text):
        chunks.append(text[s:s+size]); s+=size-overlap
    return chunks

chunks = chunk_text(full_text, 800, 150)
print(f"Total chunks: {len(chunks)}")
print(f"\nOne sample chunk:\n{chunks[0][:800]}\n")

# STEP 3 & 4: Embeddings and Search
model = SentenceTransformer('all-MiniLM-L6-v2')
emb = model.encode(chunks)

def search(query, embeddings, chunk_list, k=3):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, embeddings)[0]
    idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i]), chunk_list[i]) for i in idx]

# STEP 5: Test search alone
print("--- SEARCH TEST ---")
for i,s,c in search("What is the AC room fee?", emb, chunks):
    print(f"Chunk {i} Score {s:.4f}: {c[:300]}...")



print("\n\n===== 3 IN-DOCUMENT QUESTIONS =====")
bot("What is the annual fee for 3-sharing AC room?") # number answer
bot("What is the fine for late entry after 11 PM?")
bot("When is the room allotment date?") # date answer

print("\n\n===== 3 OUT-OF-DOCUMENT QUESTIONS =====")
bot("What is the fee for 2-sharing room in Harvard?")
bot("Who is the principal of SRM University?")
bot("What is the hostel fee for boys hostel?")

# EXPERIMENT - Chunk 3000
print("\n\n===== EXPERIMENT: CHUNK SIZE 3000 =====")
chunks_big = chunk_text(full_text, 3000, 150)
emb_big = model.encode(chunks_big)
print(f"Old chunks 800: {len(chunks)} | New chunks 3000: {len(chunks_big)}")

for q in ["What is the annual fee for 3-sharing AC room?", "What is the fine for late entry after 11 PM?", "When is the room allotment date?"]:
    bot(q, emb_big, chunks_big)

PDF created
Total characters: 432
Total chunks: 1

One sample chunk:
SRM Hostel Rulebook 2025-26 - Kaveri Block-C. Valid June 1 2025 to May 31 2026. Admission fee Rs. 5,000 no
Hostel Timings: Gate 5:00 AM to 9:30 PM weekdays, 10:00 PM weekends. Biometric 9:15-9:30 PM daily. Miss 3
Rules: Ragging strictly prohibited. Fine Rs. 10,000 and expulsion. Smoking, alcohol, drugs prohibited. Room insp
Outpass: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warde




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- SEARCH TEST ---
Chunk 0 Score 0.2443: SRM Hostel Rulebook 2025-26 - Kaveri Block-C. Valid June 1 2025 to May 31 2026. Admission fee Rs. 5,000 no
Hostel Timings: Gate 5:00 AM to 9:30 PM weekdays, 10:00 PM weekends. Biometric 9:15-9:30 PM daily. Miss 3
Rules: Ragging strictly prohibited. Fine Rs. 10,000 and expulsion. Smoking, alcohol, dr...


===== 3 IN-DOCUMENT QUESTIONS =====


NameError: name 'bot' is not defined

In [1]:
!pip install -q pymupdf sentence-transformers scikit-learn google-genai

import fitz

doc = fitz.open()
page = doc.new_page(width=595, height=842)

t = """
SRM Hostel Rulebook 2025

AC Room Fee: The AC room fee is Rs. 85,000 per year extra over normal room. Total fee for AC room is Rs. 1,85,000 per year.

Outpass: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warden approval. Medical outpass allowed 24/7.

Mess Timings: Breakfast 7:30-9:00 AM, Lunch 12:30-2:30 PM, Dinner 7:30-9:30 PM. Non-veg served on Wednesday and Sunday.

Hostel In-time: Boys hostel in-time is 9:30 PM, Girls hostel in-time is 8:30 PM.

Laundry: Free laundry service twice a week - Monday and Thursday.

WIFI: High speed WIFI available 24/7 with 50 Mbps limit per user.
"""

page.insert_text((50,50), t, fontsize=11, fontname="helv")
doc.save("/content/SRM_Hostel_Rulebook_2025.pdf")
print("PDF created - Total chars:", len(t))

PDF created - Total chars: 636


In [12]:
!pip install -q pymupdf sentence-transformers google-generativeai

In [28]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def search(q, embeddings, clist, k=3):
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity
    q_emb = model.encode([q])
    scores = cosine_similarity(q_emb, embeddings)[0]
    top_idx = scores.argsort()[-k:][::-1]
    return [(i, scores[i], clist[i]) for i in top_idx]

def bot(q, embeddings=emb, clist=chunks):
    top = search(q, embeddings, clist, 3)
    context = "\n\n".join([c for _,_,c in top])
    prompt = f"You are hostel assistant. Answer ONLY from context. If not in context say 'I cannot find the answer in the document.'\n\nContext:\n{context}\n\nQ: {q}\nA:"
    resp = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    ans = resp.text
    print(f"\nQ: {q}\nA: {ans}")
    for idx, score, _ in top:
        print(f" -> From Chunk {idx} Score {score:.4f}")
    print("-"*50)
    return ans

print("Bot Ready! ✅")

Bot Ready! ✅


In [ ]:
!pip install -q -U google-genai

In [26]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def bot(q, embeddings=emb, clist=chunks):
    top = search(q, embeddings, clist, 3)
    context = "\n\n".join([c for _,_,c in top])
    prompt = f"You are hostel assistant. Answer ONLY from context. If not in context say 'I cannot find the answer in the document.'\n\nContext:\n{context}\n\nQ: {q}\nA:"
    resp = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    ans = resp.text
    print(f"\nQ: {q}\nA: {ans}")
    for idx, score, _ in top:
        print(f"  -> From Chunk {idx} Score {score:.4f}")
    print("-"*50)
    return ans

print("Bot Ready! ✅")

Bot Ready! ✅


In [29]:
bot("What is the annual fee for 3-sharing AC room?", emb, chunks)
bot("What is the fine for late entry after 11 PM?", emb, chunks)
bot("When is the room allotment date?", emb, chunks)
bot("What is the fee for 2-sharing room in Harvard?", emb, chunks)
bot("Who is the principal of SRM University?", emb, chunks)
bot("What is the hostel fee for boys hostel?", emb, chunks)


Q: What is the annual fee for 3-sharing AC room?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.2356
--------------------------------------------------

Q: What is the fine for late entry after 11 PM?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.3267
--------------------------------------------------

Q: When is the room allotment date?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.4036
--------------------------------------------------

Q: What is the fee for 2-sharing room in Harvard?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.2696
--------------------------------------------------

Q: Who is the principal of SRM University?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.4339
--------------------------------------------------

Q: What is the hostel fee for boys hostel?
A: I cannot find the answer in the document.
 -> From Chunk 0 Score 0.4401
--------------------

'I cannot find the answer in the document.'

In [2]:
!pip install -q pymupdf sentence-transformers scikit-learn google-genai

import fitz

doc = fitz.open()
page = doc.new_page(width=595, height=842)

t = """
SRM Hostel Rulebook 2025

AC Room Fee: The AC room fee is Rs. 85,000 per year extra over normal room. Total fee for AC room is Rs. 1,85,000 per year.

Outpass: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warden approval. Medical outpass allowed 24/7.

Mess Timings: Breakfast 7:30-9:00 AM, Lunch 12:30-2:30 PM, Dinner 7:30-9:30 PM. Non-veg served on Wednesday and Sunday.

Hostel In-time: Boys hostel in-time is 9:30 PM, Girls hostel in-time is 8:30 PM.

Laundry: Free laundry service twice a week - Monday and Thursday.

WIFI: High speed WIFI available 24/7 with 50 Mbps limit per user.
"""

page.insert_text((50,50), t, fontsize=11, fontname="helv")
doc.save("/content/SRM_Hostel_Rulebook_2025.pdf")
print("PDF created - Total chars:", len(t))

PDF created - Total chars: 636


In [3]:
import fitz
from sentence_transformers import SentenceTransformer
import numpy as np

pdf = fitz.open("/content/SRM_Hostel_Rulebook_2025.pdf")
full_text = "".join([p.get_text() for p in pdf])
print(f"Total characters: {len(full_text)}")

def chunk_text(text, size=800, overlap=150):
    chunks=[]; s=0
    while s < len(text):
        chunks.append(text[s:s+size]); s+=size-overlap
    return chunks

chunks = chunk_text(full_text, 800, 150)
print(f"Total chunks: {len(chunks)}")
print(f"Sample:\n{chunks[0][:500]}")

model = SentenceTransformer('all-MiniLM-L6-v2')
emb = model.encode(chunks, batch_size=8, show_progress_bar=True)
emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
print(f"Emb shape: {emb.shape} ✅")

Total characters: 557
Total chunks: 1
Sample:
SRM Hostel Rulebook 2025
AC Room Fee: The AC room fee is Rs. 85,000 per year extra over normal room. Total fee for AC room is Rs. 1,8
Outpass: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warde
Mess Timings: Breakfast 7:30-9:00 AM, Lunch 12:30-2:30 PM, Dinner 7:30-9:30 PM. Non-veg served on Wedne
Hostel In-time: Boys hostel in-time is 9:30 PM, Girls hostel in-time is 8:30 PM.
Laundry: Free laundry service twice a week - Monday and Thursday.
WIFI: Hig


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Emb shape: (1, 384) ✅


In [4]:
from google import genai
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def search(q, k=3):
    q_emb = model.encode([q])
    q_emb = q_emb / np.linalg.norm(q_emb)
    scores = cosine_similarity(q_emb, emb)[0]
    idx = scores.argsort()[-k:][::-1]
    return [(i, scores[i], chunks[i]) for i in idx]

def bot(q):
    top = search(q, 3)
    context = "\n\n".join([c for _,_,c in top])
    prompt = f"You are hostel assistant. Answer ONLY from context. If not in context say 'I cannot find the answer in the document.'\n\nContext:\n{context}\n\nQ: {q}"
    resp = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)
    ans = resp.text
    print(f"\nQ: {q}\nA: {ans}")
    for i,s,_ in top:
        print(f" -> Chunk {i} Score {s:.4f}")
    print("-"*50)
    return ans

print("Bot Ready ✅")

Bot Ready ✅


In [5]:
bot("What is the AC room fee?")
bot("What is the outpass rule?")
bot("What are mess timings?")
bot("What is the capital of France?") # Out of doc - should say cannot find
bot("Who is the Prime Minister?")
bot("Explain quantum physics?")


Q: What is the AC room fee?
A: The AC room fee is Rs. 85,000 per year extra over normal room. Total fee for AC room is Rs. 1,8
 -> Chunk 0 Score 0.5807
--------------------------------------------------

Q: What is the outpass rule?
A: Local outpass allowed weekends with parent consent via app. Emergency outpass anytime with warde
 -> Chunk 0 Score 0.2770
--------------------------------------------------

Q: What are mess timings?
A: Breakfast 7:30-9:00 AM, Lunch 12:30-2:30 PM, Dinner 7:30-9:30 PM
 -> Chunk 0 Score 0.0297
--------------------------------------------------

Q: What is the capital of France?
A: I cannot find the answer in the document.
 -> Chunk 0 Score 0.0621
--------------------------------------------------

Q: Who is the Prime Minister?
A: I cannot find the answer in the document.
 -> Chunk 0 Score -0.0078
--------------------------------------------------

Q: Explain quantum physics?
A: I cannot find the answer in the document.
 -> Chunk 0 Score 0.0347
-----------

'I cannot find the answer in the document.'